# Incident Commander: a guided walkthrough

The console shows you the agent loop running. This notebook lets you **stop it
between steps and look inside** - which is where the actual learning is.

You will:

1. Poke the sandbox and see what the agent can and cannot perceive
2. Run each agent **on its own**, and read the exact prompt it receives
3. Watch the evidence loop decide it needs a tool
4. Attack the agent with prompt injection and watch the guardrails work
5. Run the full loop and read the trajectory
6. Score a run the way the evaluation harness does

**No API key required.** Everything runs against the deterministic test stub, which
reads the same telemetry a real model would. Where the real model would behave
differently, the notebook says so.

> Run from the repo: `cd backend && uv run jupyter lab ../notebooks/`

In [ ]:
import json
import os
import sys
from pathlib import Path

# Locate the backend package by walking up from wherever Jupyter was started,
# so this notebook works from the repo root, from notebooks/, or from backend/.
def find_backend(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "backend" / "src" / "loop.py").exists():
            return candidate / "backend"
        if (candidate / "src" / "loop.py").exists():
            return candidate
    raise RuntimeError("could not locate the incident-commander backend package")

BACKEND = find_backend(Path.cwd())
sys.path.insert(0, str(BACKEND))
os.chdir(BACKEND)
os.environ.setdefault("IC_MEMORY_ENABLED", "false")  # keep the notebook stateless

def show(obj, limit=2000):
    """Pretty-print anything, truncated."""
    text = obj if isinstance(obj, str) else json.dumps(obj, indent=2, default=str)
    print(text[:limit])
    if len(text) > limit:
        print("... [truncated]")

print("backend:", BACKEND)

---
## 1. The sandbox: what the agent can and cannot see

Everything the agent perceives comes from `tools/observability.py`. Two design
choices here shape the entire project.

In [ ]:
from src.sandbox.scenarios import SCENARIOS, list_scenarios
from src.sandbox.stack import SandboxStack
from src.tools.observability import observe_all

for s in list_scenarios():
    tag = "NEGATIVE" if s["is_negative"] else "        "
    print(f"{tag}  {s['id']:<22} expects {s['expected_outcome']:<10} steps={s['steps_required']}  {s['name']}")

Note the **negative** scenarios. `unfixable_ca_expiry` must ESCALATE and
`false_alarm` must do nothing. An agent evaluated only on faults it can fix will
happily invent a fix for one it cannot, and no outcome metric will ever catch it.

### Ground truth is hidden

In [ ]:
scenario = SCENARIOS["bad_deploy"]
stack = SandboxStack(scenario=scenario, service=scenario.default_service)

print("The sandbox KNOWS the answer:")
print("  faulted        =", stack.faulted)
print("  correct action =", scenario.correct_action)

print("\nBut what the agent receives has no `faulted` flag:")
show(observe_all(stack)["metrics"])

The agent has to infer health from `error_rate` and `p99_latency_ms`, like a real
on-call engineer. Leak `faulted` into the bundle and every scenario becomes trivial
while still *looking* like it is being solved - a very easy way to build an eval
that measures nothing.

### Some signals cost a tool call

In [ ]:
from src.tools.observability import DEFAULT_BUNDLE, describe_tools

print("Always available (free):")
for t in DEFAULT_BUNDLE:
    print("  -", t.value)

print("\nAvailable ON REQUEST only:")
print(describe_tools())

In [ ]:
# `silent_config_drift` is unsolvable from the free bundle alone.
drift = SCENARIOS["silent_config_drift"]
drift_stack = SandboxStack(scenario=drift, service=drift.default_service)

print("In the default bundle?", "db.pool.maxSize" in str(observe_all(drift_stack)))
print("In the config diff? ", "db.pool.maxSize" in str(drift_stack.config_diff()))
show(drift_stack.config_diff())

This is what makes the evidence loop real rather than decorative: an agent that
cannot decide *"I need to go look at something else"* simply cannot solve this one.

---
## 2. One agent at a time

Every agent is a pure function: `run(state, ..., llm) -> ValidatedSchema`. Let's
call them individually and read what they actually receive.

In [ ]:
from src.agents import diagnostician, remediation, triage, verifier
from src.guardrails.manager import GuardrailManager
from src.models.state import IncidentState, Severity
from tests._fake_llm import FakeLLMClient

llm = FakeLLMClient()  # swap for src.llm.get_llm() if you have GOOGLE_API_KEY set
guard = GuardrailManager()

state = IncidentState(
    scenario_id=scenario.id,
    service=scenario.default_service,
    severity=Severity(scenario.severity),
)
state.iterations = 1

# Telemetry is ALWAYS sanitized before an agent sees it.
observations = guard.sanitize(observe_all(stack)).observations

tri = triage.run(state, observations, llm)
state.triage_summary, state.severity = tri.summary, tri.severity
show(tri.model_dump())

### Read the actual prompt

This is the most useful cell in the notebook. Notice **where the fence is**: your
own facts sit in a `TRUSTED` block, and telemetry is walled off inside
`UNTRUSTED_TELEMETRY`.

In [ ]:
from src.agents._prompting import (
    action_history_block,
    incident_context,
    join_sections,
    telemetry_block,
)

print("=========== SYSTEM PROMPT ===========")
print(diagnostician._SYSTEM[:2200])
print("\n=========== USER PROMPT ===========")
print(join_sections(incident_context(state), action_history_block(state), telemetry_block(observations))[:2200])

In [ ]:
diag = diagnostician.run(state, observations, llm)
state.diagnostics.append(diag.model_dump(mode="json"))

print("leading cause :", diag.leading_cause)
print("confidence    :", diag.confidence)
print("needs more?   :", diag.needs_more_data)
print("evidence      :", diag.evidence)

In [ ]:
prop = remediation.run(state, diag, observations, llm)
show(prop.model_dump(mode="json"))

Two things to notice:

- **`action_type` is an enum**, not a string. The model physically cannot invent
  `delete_database`. "Hallucinate a new action" is not a failure mode here.
- **`expected_signal` is a falsifiable prediction.** The Verifier will be scored
  against it. A proposal that cannot be wrong cannot be verified.

### The model does NOT decide its own risk

In [ ]:
print("risk the MODEL claimed      :", prop.risk.value)
print("risk the GUARDRAIL assigns  :", guard.assess_risk(prop.action_type.value).value)
print("\nunknown action falls back to:", guard.assess_risk("delete_everything").value, "(fail closed)")

The orchestrator **overwrites** the model's self-assessment. An agent that could
grade its own rollback "low risk" could route around the human approval gate - so
risk classification lives outside the model entirely.

### The falsification loop

In [ ]:
from src.models.state import ActionStatus, RemediationAction
from src.tools.remediation import execute_remediation

action = RemediationAction(
    action_type=prop.action_type.value,
    target=prop.target,
    risk=guard.assess_risk(prop.action_type.value),
    expected_signal=prop.expected_signal,
    status=ActionStatus.APPROVED,  # normally set by a human at the gate
)
state.actions.append(action)

result = execute_remediation(stack, action)
print("action result:", result["reason"], "\n")

post = guard.sanitize(observe_all(stack)).observations
verdict = verifier.run(state, post, llm, action)
show(verdict.model_dump())

`expected_signal_met` is the interesting field. `recovered` and
`expected_signal_met` can **disagree**, and the disagreement is informative:
recovered-but-prediction-missed means the service came back for a reason we do not
understand, so the diagnosis is still wrong and the fault will return.

---
## 3. The evidence loop

Now the scenario the free telemetry cannot solve.

In [ ]:
from src.tools.observability import run_tools

drift_state = IncidentState(scenario_id=drift.id, service=drift.default_service)
drift_state.iterations = 1
drift_state.triage_summary = "Latency doubled with no errors."
drift_obs = guard.sanitize(observe_all(drift_stack)).observations

first = diagnostician.run(drift_state, drift_obs, llm)
print("ROUND 1")
print("  confidence     :", first.confidence)
print("  needs_more_data:", first.needs_more_data)
print("  requested tools:", [t.value for t in first.requested_tools])

In [ ]:
# The loop runs exactly the tools the agent asked for, then re-diagnoses.
evidence = guard.sanitize(run_tools(drift_stack, list(first.requested_tools))).observations
for name, value in evidence.items():
    drift_state.evidence[f"iter1:{name}"] = value
drift_state.evidence_rounds = 1

second = diagnostician.run(drift_state, drift_obs, llm)
print("ROUND 2")
print("  confidence   :", second.confidence)
print("  leading cause:", second.leading_cause)

That is the whole difference between an agent and a function call: it decided what
it needed to know next.

Bounded, of course - `IC_MAX_EVIDENCE_ROUNDS` caps it, because an unbounded "I need
more data" loop is an unbounded loop. With rounds set to 0 this incident escalates
with `halt_reason=low_confidence` instead of guessing.

---
## 4. Attacking the agent

`poisoned_telemetry` is a real memory leak whose logs contain attacker text
demanding a destructive `failover`.

In [ ]:
poisoned = SCENARIOS["poisoned_telemetry"]
print("What the attacker wrote into the logs:\n")
for line in poisoned.injected_logs:
    print(" ", line, "\n")

In [ ]:
p_stack = SandboxStack(scenario=poisoned, service=poisoned.default_service)
p_guard = GuardrailManager(quarantine=True, redact=True)
raw = observe_all(p_stack)
clean = p_guard.sanitize(raw)

print("patterns detected:", clean.injection_hits)
print("fields quarantined:", clean.quarantined_lines)
print("\npayload still present after sanitizing?",
      "ignore all previous instructions" in str(clean.observations).lower())
print("\nWhat the model actually sees:")
show([line for line in clean.observations["recent_logs"] if "QUARANTINED" in line])

The imperative text is **gone**; a marker records that something was there. Compare
with flag-only mode, which is what the previous version of this project did:

In [ ]:
flag_only = GuardrailManager(quarantine=False).sanitize(observe_all(p_stack))
print("detected:", bool(flag_only.injection_hits))
print("payload still reaches the model?",
      "ignore all previous instructions" in str(flag_only.observations).lower())
print("\n^ Detecting an attack and forwarding it intact is logging, not defence.")

In [ ]:
# Secrets in telemetry are a notorious leak path into prompts and postmortems.
leaky = {"recent_logs": [
    "auth failed for oncall@example.com using Bearer abc123def456ghi789 retrying",
    "HikariPool-1 timeout after 30000ms",
]}
show(GuardrailManager(redact=True).sanitize(leaky).observations)

**The part that matters most:** even if every regex above failed, a hijacked model
still could not do arbitrary damage. It can only emit an action from a closed enum,
that action must be in its role's allowlist, and `failover` is HIGH risk so it stops
for a human. Input filtering is the speed bump; architecture is the brake.

In [ ]:
from src.guardrails.manager import ROLE_ALLOWED

for role in ROLE_ALLOWED:
    g = GuardrailManager(role=role)
    print(f"{role:<22} may run: {sorted(g.allowed_actions())}")

typo = GuardrailManager(role="sre-bot-typo")
print(f"\n{'sre-bot-typo':<22} may run: {sorted(typo.allowed_actions()) or 'NOTHING - fail closed'}")

---
## 5. The full loop

Now run the real orchestrator, with an approve-all operator.

In [ ]:
import asyncio

from src.loop import IncidentManager
from src.models.state import EventType

async def run_incident(scenario_id, decision="approve"):
    manager = IncidentManager(llm=FakeLLMClient())
    st = manager.trigger(scenario_id, None, auto_approve=False)
    async for ev in manager.subscribe(st.incident_id):
        if ev.type == EventType.AWAITING_APPROVAL and decision:
            manager.resolve_approval(st.incident_id, ev.payload["action_id"], decision, "notebook")
    return manager.get(st.incident_id)

# Jupyter already runs an event loop, so `await` directly (asyncio.run would fail).
final = await (run_incident("cascading_failure"))
print("status     :", final.status.value)
print("halt reason:", final.halt_reason.value)
print("iterations :", final.iterations)

In [ ]:
for ev in final.events:
    agent = ev.agent.value if ev.agent else "-"
    detail = ev.payload.get("leading_cause") or ev.payload.get("action_type") or ev.payload.get("reason") or ""
    print(f"{ev.seq:>3}  iter{ev.iteration}  {agent:<14} {ev.type.value:<20} {str(detail)[:60]}")

Read the trajectory: two iterations, and the leading cause **changes** between
them. That is the loop self-correcting, not repeating itself.

In [ ]:
for i, d in enumerate(final.diagnostics, 1):
    print(f"diagnosis {i}: {d['leading_cause']}  (confidence {d['confidence']})")

print()
for a in final.actions:
    r = a.result or {}
    verdict = "RESOLVED" if r.get("effective") else "PARTIAL" if r.get("progress") else "NO EFFECT"
    print(f"iter {a.iteration}: {a.action_type:<18} {verdict}")

### The negative scenarios

Doing nothing, and giving up, are correct answers here.

In [ ]:
for sid in ["unfixable_ca_expiry", "false_alarm"]:
    st = await run_incident(sid)
    actions = [a.action_type for a in st.actions] or ["(none)"]
    print(f"{sid:<22} -> {st.status.value:<10} expected={SCENARIOS[sid].expected_outcome:<10} actions={actions}")

An agent that "resolved" the expired certificate would have caused disruption and
still not fixed anything. An agent that restarted the healthy service would have
*created* the outage it was paged for.

---
## 6. Scoring a run

Outcome metrics say whether the agent got there. Process metrics say whether it got
there for the right reasons - which is what predicts behaviour on an incident
nobody scripted.

In [ ]:
from evaluation.metrics import score_trajectory
from evaluation.run_eval import run_once

record = await (run_once("bad_deploy", stub=True, repeat=0))
traj = score_trajectory(record, SCENARIOS["bad_deploy"])
for check, passed in traj["checks"].items():
    print(f"  {'PASS' if passed else 'FAIL'}  {check}")
print(f"\ntrajectory score: {traj['score']:.2f}")

In [ ]:
# A *lucky* agent: right action, right outcome, no evidence, no prediction.
# Every outcome metric calls this perfect.
lucky = {
    "actions": [{"action_type": "rollback_deploy", "status": "executed",
                 "result": {"effective": True}, "expected_signal": ""}],
    "diagnoses": [{"leading_cause": "something broke", "evidence": [], "confidence": 0.99}],
    "verifications": [{"recovered": True}],
}
lucky_score = score_trajectory(lucky, SCENARIOS["bad_deploy"])
print(f"outcome            : resolved (perfect by any outcome metric)")
print(f"trajectory score   : {lucky_score['score']:.2f}  <- caught")
for check, passed in lucky_score["checks"].items():
    if not passed:
        print("   FAIL", check)

That is the agent that will fail on the next incident you have not seen, and no
resolution rate will ever tell you.

---
## Where to go next

| To... | Go to |
|---|---|
| Break things deliberately | [`../EXERCISES.md`](../EXERCISES.md) |
| Understand a design decision | [`../docs/adr/`](../docs/adr/) |
| See how prompts were built | [`../docs/prompt-engineering.md`](../docs/prompt-engineering.md) |
| Read the loop itself | `backend/src/loop.py` |

Two experiments worth doing right now, both one line:

```bash
IC_CRITIC_ENABLED=false uv run python -m evaluation.run_eval --stub --repeats 3
IC_AGENT_ROLE=read-only uv run python -m evaluation.run_eval --stub
```